# Boltz1 3D Visualization Template

Generic template for creating interactive 3D visualizations of Boltz1 predictions.

## Usage Instructions
1. Copy this template to your demo folder
2. Update the file paths in the second cell
3. Customize the visualization functions for your specific system
4. Run all cells to generate interactive 3D views

## Requirements
- py3Dmol (pip install py3Dmol)
- matplotlib, seaborn, numpy
- Jupyter notebook environment

In [ ]:
# Import required libraries
import py3Dmol
import json
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from Bio import PDB
import warnings
warnings.filterwarnings('ignore')

print("✅ Libraries imported successfully")
print(f"📁 Working directory: {Path.cwd()}")

In [ ]:
# UPDATE THESE PATHS for your specific demo
structure_path = "output/YOUR_RESULTS_FOLDER/predictions/YOUR_PREDICTION/YOUR_MODEL.cif"
confidence_path = "output/YOUR_RESULTS_FOLDER/predictions/YOUR_PREDICTION/confidence_YOUR_MODEL.json"

# Check if files exist
if Path(structure_path).exists():
    print(f"✅ Structure file found: {structure_path}")
else:
    print(f"❌ Structure file not found: {structure_path}")
    print("Please update the structure_path variable above")

if Path(confidence_path).exists():
    print(f"✅ Confidence file found: {confidence_path}")
    # Load confidence data
    with open(confidence_path, 'r') as f:
        confidence_data = json.load(f)
    print(f"📊 Overall confidence: {confidence_data['confidence_score']*100:.1f}%")
else:
    print(f"❌ Confidence file not found: {confidence_path}")
    print("Please update the confidence_path variable above")

## Basic 3D Structure Visualization

In [ ]:
# Basic 3D visualization function
def create_basic_3d_visualization(structure_path, width=800, height=600):
    """Create basic interactive 3D visualization"""
    
    # Initialize viewer
    view = py3Dmol.view(width=width, height=height)
    
    # Load structure
    with open(structure_path, 'r') as f:
        cif_data = f.read()
    
    view.addModel(cif_data, 'cif')
    
    # Basic styling - customize as needed
    view.setStyle({}, {'cartoon': {'color': 'spectrum', 'opacity': 0.8}})
    
    # Add confidence label if available
    if Path(confidence_path).exists():
        conf_score = confidence_data['confidence_score'] * 100
        view.addLabel(f'Confidence: {conf_score:.1f}%', {
            'position': {'x': 0, 'y': 15, 'z': 0}, 
            'backgroundColor': 'white', 
            'fontColor': 'black'
        })
    
    view.zoomTo()
    view.spin(False)
    
    return view

# Create and display basic visualization
if Path(structure_path).exists():
    print("🎨 Creating basic 3D visualization...")
    basic_viewer = create_basic_3d_visualization(structure_path)
    basic_viewer.show()
    print("✅ Basic 3D visualization created!")
else:
    print("❌ Cannot create visualization - check file paths")

## Confidence-Colored Visualization

In [ ]:
# Confidence-colored visualization
def create_confidence_visualization(structure_path, width=800, height=600):
    """Create visualization colored by confidence scores"""
    
    view = py3Dmol.view(width=width, height=height)
    
    # Load structure
    with open(structure_path, 'r') as f:
        cif_data = f.read()
    
    view.addModel(cif_data, 'cif')
    
    # Color by B-factor (confidence)
    view.setStyle({}, {
        'cartoon': {
            'colorscheme': {
                'prop': 'b',
                'gradient': 'RdYlBu',  # Red-Yellow-Blue
                'min': 50,
                'max': 100
            },
            'opacity': 0.8
        }
    })
    
    # Add confidence legend
    view.addLabel('High Confidence', {'position': {'x': -15, 'y': 10, 'z': 0}, 'backgroundColor': 'blue', 'fontColor': 'white'})
    view.addLabel('Medium Confidence', {'position': {'x': -15, 'y': 5, 'z': 0}, 'backgroundColor': 'yellow', 'fontColor': 'black'})
    view.addLabel('Low Confidence', {'position': {'x': -15, 'y': 0, 'z': 0}, 'backgroundColor': 'red', 'fontColor': 'white'})
    
    view.zoomTo()
    
    return view

if Path(structure_path).exists():
    print("🎨 Creating confidence-colored visualization...")
    conf_viewer = create_confidence_visualization(structure_path)
    conf_viewer.show()
    print("✅ Confidence visualization created!")
else:
    print("❌ Cannot create confidence visualization - check file paths")

## Surface Representation

In [ ]:
# Surface representation
def create_surface_visualization(structure_path, width=800, height=600):
    """Create molecular surface visualization"""
    
    view = py3Dmol.view(width=width, height=height)
    
    # Load structure
    with open(structure_path, 'r') as f:
        cif_data = f.read()
    
    view.addModel(cif_data, 'cif')
    
    # Show cartoon (transparent) + surface
    view.setStyle({}, {'cartoon': {'color': 'gray', 'opacity': 0.3}})
    
    # Add molecular surface
    view.addSurface(py3Dmol.VDW, {
        'opacity': 0.7,
        'colorscheme': 'hydrophobicity'
    })
    
    view.zoomTo()
    
    return view

if Path(structure_path).exists():
    print("🌊 Creating surface visualization...")
    surface_viewer = create_surface_visualization(structure_path)
    surface_viewer.show()
    print("✅ Surface visualization created!")
else:
    print("❌ Cannot create surface visualization - check file paths")

## Analysis and Summary

In [ ]:
# Analysis function
def analyze_prediction():
    """Analyze prediction quality and generate summary"""
    
    if not Path(confidence_path).exists():
        print("❌ Cannot analyze - confidence file not found")
        return
    
    print("📊 PREDICTION ANALYSIS")
    print("=" * 30)
    
    overall = confidence_data['confidence_score'] * 100
    ptm = confidence_data.get('ptm', 0) * 100
    iptm = confidence_data.get('iptm', 0) * 100
    
    print(f"Overall Confidence: {overall:.1f}%")
    if ptm > 0:
        print(f"Protein Structure (PTM): {ptm:.1f}%")
    if iptm > 0:
        print(f"Interface Quality (iPTM): {iptm:.1f}%")
    
    # Quality assessment
    if overall >= 90:
        quality = "Excellent - Near experimental accuracy"
    elif overall >= 80:
        quality = "Very Good - High confidence"
    elif overall >= 70:
        quality = "Good - Reliable structure"
    elif overall >= 60:
        quality = "Moderate - Use with caution"
    else:
        quality = "Poor - Not recommended"
    
    print(f"\nQuality Assessment: {quality}")
    
    # Create simple visualization
    plt.figure(figsize=(8, 6))
    
    metrics = ['Overall']
    scores = [overall]
    
    if ptm > 0:
        metrics.append('PTM')
        scores.append(ptm)
    if iptm > 0:
        metrics.append('iPTM')
        scores.append(iptm)
    
    bars = plt.bar(metrics, scores, color=['lightblue', 'lightgreen', 'orange'][:len(metrics)], 
                   alpha=0.7, edgecolor='black')
    
    # Add value labels
    for bar, score in zip(bars, scores):
        plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                f'{score:.1f}%', ha='center', va='bottom', fontweight='bold')
    
    plt.ylabel('Confidence Score (%)', fontweight='bold')
    plt.title('Prediction Quality Metrics', fontweight='bold')
    plt.ylim(0, 100)
    plt.grid(True, alpha=0.3)
    
    # Add quality threshold lines
    plt.axhline(y=90, color='green', linestyle='--', alpha=0.7, label='Excellent (>90%)')
    plt.axhline(y=70, color='orange', linestyle='--', alpha=0.7, label='Good (>70%)')
    plt.legend()
    
    plt.tight_layout()
    plt.show()

# Run analysis
try:
    analyze_prediction()
    print("✅ Analysis complete")
except Exception as e:
    print(f"⚠️ Analysis error: {e}")

## Customization Guide

### For Different System Types:

**Single Proteins:**
- Use spectrum coloring: `'color': 'spectrum'`
- Focus on secondary structure
- Highlight confidence variations

**Protein Complexes:**
- Color different chains: `{'chain': 'A'}`, `{'chain': 'B'}`
- Show interface surfaces
- Highlight binding regions

**Protein-Ligand Systems:**
- Use cartoon for protein: `'cartoon': {...}`
- Use stick for ligands: `'stick': {...}`
- Show binding site surfaces

### Color Schemes:
- `'spectrum'`: Rainbow N→C terminus
- `'RdYlBu'`: Red-Yellow-Blue confidence
- `'hydrophobicity'`: Hydrophobic/hydrophilic
- `'ss'`: Secondary structure

### Styling Options:
- `'cartoon'`: Ribbon representation
- `'stick'`: Bond representation
- `'sphere'`: Space-filling
- `'surface'`: Molecular surface

### Interactive Controls:
- **Left Click + Drag**: Rotate
- **Right Click + Drag**: Pan
- **Scroll**: Zoom
- **Double Click**: Center

### Export Options:
- Right-click → Save image
- Screenshot for presentations
- PNG format recommended

This template provides a foundation for creating interactive 3D visualizations of any Boltz1 prediction. Customize the functions and styling based on your specific biological system and presentation needs.